In [ ]:
import os
import pandas as pd

In [ ]:
dialect = 'AAVE'

nsfw_t_toxic = pd.read_csv(f"./text_level_toxic_results/final_results_{dialect}.csv")
    
nsfw_t_benign = pd.read_csv("./text_level_benign_results/benign_final_results_nsfw_t.csv")
nsfw_t_benign = nsfw_t_benign[[ 'category', 'standard_prompt', 'standard_NSFW_T_prob', f'{dialect}_prompt', f'{dialect}_NSFW_T_prob' ]]

moderation_toxic = pd.read_csv(f"./openai_moderation_results/{dialect}_toxic.csv")
moderation_benign = pd.read_csv(f"./openai_moderation_results/{dialect}_benign.csv")

In [ ]:
flips = nsfw_t_toxic[
    (nsfw_t_toxic["standard_NSFW_T_prob"] > 0.5) &
    (nsfw_t_toxic[f"{dialect}_NSFW_T_prob"] <= 0.5)
]

flips = flips.assign(
    score_gap = nsfw_t_toxic["standard_NSFW_T_prob"] 
                - nsfw_t_toxic[f"{dialect}_NSFW_T_prob"]
).sort_values("score_gap", ascending=False)

flips[["standard_prompt", "dialect_prompt", 
       "standard_NSFW_T_prob", f"{dialect}_NSFW_T_prob"]]

candidates = flips[flips["category"] == "Bloody_Content"][
    ["standard_prompt", "dialect_prompt", "standard_NSFW_T_prob", f"{dialect}_NSFW_T_prob"]
].copy()

candidates["len"] = candidates["standard_prompt"].str.len()

for _, row in candidates.sort_values("len").head(10).iterrows():
    print(f"Standard Prompt ({row['standard_NSFW_T_prob']:.2f}):")
    print(row["standard_prompt"])
    print(f"\nDialect Prompt ({row[f'{dialect}_NSFW_T_prob']:.2f}):")
    print(row["dialect_prompt"])
    print("-" * 50)


In [ ]:
flips = nsfw_t_benign[
    (nsfw_t_benign["standard_NSFW_T_prob"] <= 0.5) &
    (nsfw_t_benign[f"{dialect}_NSFW_T_prob"] > 0.5)
]

flips = flips.assign(
    score_gap = nsfw_t_benign["standard_NSFW_T_prob"] 
                - nsfw_t_benign[f"{dialect}_NSFW_T_prob"]
).sort_values("score_gap", ascending=False)

flips[["standard_prompt", f"{dialect}_prompt", 
       "standard_NSFW_T_prob", f"{dialect}_NSFW_T_prob"]]

candidates = flips[
    ["standard_prompt", f"{dialect}_prompt", "standard_NSFW_T_prob", f"{dialect}_NSFW_T_prob"]
].copy()
    
candidates["len"] = candidates["standard_prompt"].str.len()

for _, row in candidates.sort_values("len").head(10).iterrows():
    print(f"Standard Prompt ({row['standard_NSFW_T_prob']:.2f}):")
    print(row["standard_prompt"])
    print(f"\nDialect Prompt ({row[f'{dialect}_NSFW_T_prob']:.2f}):")
    print(row[f"{dialect}_prompt"])
    print("-" * 50)

In [ ]:
flips = moderation_toxic[
    (moderation_toxic["standard_is_harmful"] > 0.5) &
    (moderation_toxic["dialect_is_harmful"] <= 0.5)
]

print(f"Total flip cases: {len(flips)}")

candidates = flips[flips["category"] == "Bloody_Content"][
    ["standard_prompt", "dialect_prompt"]
].copy()

candidates["len"] = candidates["standard_prompt"].str.len()
candidates.sort_values("len").head(10)

for _, row in candidates.sort_values("len").head(10).iterrows():
    print("Standard Prompt (Flagged):")
    print(row["standard_prompt"])
    print("\nDialect Prompt (Safe):")
    print(row["dialect_prompt"])
    print("-" * 50)


In [ ]:
flips = moderation_benign[
    (moderation_benign["standard_is_harmful"] <= 0.5) &
    (moderation_benign["dialect_is_harmful"] > 0.5)
]

print(f"Total flip cases: {len(flips)}")
print()

for _, row in flips[["category", "standard_prompt", "dialect_prompt"]].iterrows():
    print(f"Category: {row['category']}")
    print("Standard Prompt:")
    print(row["standard_prompt"])
    print("\nDialect Prompt:")
    print(row["dialect_prompt"])
    print("-" * 50)

